In [1]:
from pathlib import Path
import re
import json
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# Repo paths (notebook lives in notebooks/, so repo root is one level up)
REPO_ROOT = Path("..").resolve()
DATA_RAW = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)


JOBS_FILE = "jobs_dataset.csv"
RESUMES_FILE = "resumes_dataset.csv"

jobs = pd.read_csv(DATA_RAW / JOBS_FILE)
resumes = pd.read_csv(DATA_RAW / RESUMES_FILE)

print("jobs shape:", jobs.shape)
print("resumes shape:", resumes.shape)


jobs shape: (1068, 7)
resumes shape: (1200, 14)


In [2]:
#Text normalization
def normalize_text(x):
    """
    Normalize general text fields:
    - NaN -> ""
    - collapse whitespace
    """
    if pd.isna(x):
        return ""
    x = str(x).replace("\u00a0", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()


In [3]:
#Skill parsing + cleanup (handles both ';' and ',')
SKILL_NOISE_SUFFIXES = [
    " basics", " fundamental", " fundamentals", " beginner", " introductory", " intro"
]

def clean_skill_token(token: str) -> str:
    """
    Normalize an individual skill token:
    - lowercase
    - trim spaces
    - remove trailing noise words like 'basics'/'fundamentals'
    """
    s = normalize_text(token).lower()

    # remove repeated spaces
    s = re.sub(r"\s+", " ", s).strip()

    # remove common suffix noise (only if it appears at the end)
    for suf in SKILL_NOISE_SUFFIXES:
        if s.endswith(suf):
            s = s[: -len(suf)].strip()

    # tiny cleanups (optional but safe)
    s = s.replace("  ", " ").strip()

    return s

def parse_skills(x):
    """
    Parse skills field into a clean list:
    - jobs use ';' separators
    - resumes use ',' separators
    Supports mixed separators safely.
    """
    if pd.isna(x):
        return []

    raw = str(x)

    # split on ; , / |
    parts = re.split(r"[;,/|]", raw)

    skills = []
    seen = set()

    for p in parts:
        s = clean_skill_token(p)
        if s and s not in seen:
            seen.add(s)
            skills.append(s)

    return skills


In [4]:
#Clean Jobs into canonical schema
jobs_clean = jobs.copy()

# Fill the single missing Title safely (no row drop)
jobs_clean["Title"] = jobs_clean["Title"].fillna("")

# Canonical fields
jobs_clean["job_id"] = jobs_clean["JobID"].astype(str)
jobs_clean["job_title"] = jobs_clean["Title"].map(normalize_text)

# Normalize ExperienceLevel (kept as text; we just normalize casing/spacing)
jobs_clean["experience_level"] = jobs_clean["ExperienceLevel"].map(normalize_text).str.lower()

# Keep YearsOfExperience as text (often ranges like '0-1')
jobs_clean["years_of_experience"] = jobs_clean["YearsOfExperience"].map(normalize_text)

# Parse skills
jobs_clean["job_skills_list"] = jobs_clean["Skills"].map(parse_skills)

# Build job_text for embeddings (title + exp + responsibilities + skills)
jobs_clean["job_text"] = (
    "Job Title: " + jobs_clean["job_title"] + ". "
    "Experience Level: " + jobs_clean["experience_level"] + ". "
    "Years of Experience: " + jobs_clean["years_of_experience"] + ". "
    "Responsibilities: " + jobs_clean["Responsibilities"].map(normalize_text) + ". "
    "Skills: " + jobs_clean["job_skills_list"].map(lambda xs: ", ".join(xs))
)

jobs_clean = jobs_clean[
    [
        "job_id",
        "job_title",
        "experience_level",
        "years_of_experience",
        "job_skills_list",
        "job_text",
    ]
].copy()

print("jobs_clean:", jobs_clean.shape)
display(jobs_clean.head(3))


jobs_clean: (1068, 6)


,job_id,job_title,experience_level,years_of_experience,job_skills_list,job_text
0,NET-F-001,.NET Developer,fresher,0-1,"[c#, vb.net, .net framework, .net core, asp.net, mvc, html, css, javascript, sql server, entity framework, linq, visual studio, git, unit testing]",Job Title: .NET Developer. Experience Level: fresher. Years of Experience: 0-1. Responsibilities: Assist in coding and debugging applications; Learn and apply .NET Framework and Core fundamentals;...
1,NET-F-002,.NET Developer,fresher,0-1,"[c#, .net framework, asp.net, razor, html, css, javascript, sql server, entity framework, nunit]",Job Title: .NET Developer. Experience Level: fresher. Years of Experience: 0-1. Responsibilities: Write simple C# programs under guidance; Support development of ASP.NET MVC applications; Implemen...
2,NET-F-003,.NET Developer,fresher,0-1,"[c#, vb.net, .net core, asp.net mvc, html, css, javascript, sql server, git]",Job Title: .NET Developer. Experience Level: fresher. Years of Experience: 0-1. Responsibilities: Contribute to development of small modules; Assist in bug fixing and debugging; Learn and implemen...


In [6]:
#Clean RESUMES into canonical schema
resumes_clean = resumes.copy()

# Create stable resume ids
resumes_clean = resumes_clean.reset_index(drop=True)
resumes_clean["resume_id"] = ["R_%04d" % i for i in range(len(resumes_clean))]

# Handle expected missing values as empty strings
resumes_clean["Current_Job_Title"] = resumes_clean["Current_Job_Title"].fillna("")
resumes_clean["Previous_Job_Titles"] = resumes_clean["Previous_Job_Titles"].fillna("")
resumes_clean["Certifications"] = resumes_clean["Certifications"].fillna("")
resumes_clean["Target_Job_Description"] = resumes_clean["Target_Job_Description"].fillna("")

# Canonical fields
resumes_clean["current_job_title"] = resumes_clean["Current_Job_Title"].map(normalize_text)
resumes_clean["experience_years"] = resumes_clean["Experience_Years"].astype(int)

resumes_clean["resume_skills_list"] = resumes_clean["Skills"].map(parse_skills)

# Education summary blob (kept readable)
resumes_clean["education_blob"] = (
    resumes_clean["Education_Level"].map(normalize_text) + " | " +
    resumes_clean["Degrees"].map(normalize_text) + " | " +
    resumes_clean["Field_of_Study"].map(normalize_text) + " | " +
    resumes_clean["Institute_Name"].map(normalize_text) + " | " +
    resumes_clean["Graduation_Year"].astype(str)
)

resumes_clean["target_job_description"] = resumes_clean["Target_Job_Description"].map(normalize_text)

# We include certifications and previous roles in resume_text (when present)
resumes_clean["resume_text"] = (
    "Current Title: " + resumes_clean["current_job_title"] + ". "
    "Previous Titles: " + resumes_clean["Previous_Job_Titles"].map(normalize_text) + ". "
    "Education: " + resumes_clean["education_blob"] + ". "
    "Experience Years: " + resumes_clean["experience_years"].astype(str) + ". "
    "Skills: " + resumes_clean["resume_skills_list"].map(lambda xs: ", ".join(xs)) + ". "
    "Certifications: " + resumes_clean["Certifications"].map(normalize_text) + ". "
    "Target Description: " + resumes_clean["target_job_description"]
)

resumes_clean = resumes_clean[
    [
        "resume_id",
        "current_job_title",
        "education_blob",
        "experience_years",
        "resume_skills_list",
        "target_job_description",
        "resume_text",
    ]
].copy()

print("resumes_clean:", resumes_clean.shape)
display(resumes_clean.head(3))


resumes_clean: (1200, 7)


,resume_id,current_job_title,education_blob,experience_years,resume_skills_list,target_job_description,resume_text
0,R_0000,,Master's | Master's in Cybersecurity | Cybersecurity | University of Pennsylvania | 2025,0,"[node.js, javascript, deep learning, statistics, sql]",Seeking a challenging role as a Software Developer where I can apply my skills and knowledge to contribute to organizational success and professional growth.,"Current Title: . Previous Titles: . Education: Master's | Master's in Cybersecurity | Cybersecurity | University of Pennsylvania | 2025. Experience Years: 0. Skills: node.js, javascript, deep lear..."
1,R_0001,Cybersecurity Engineer,Bachelor's | Bachelor's in Electronics Engineering | Electronics Engineering | Pune University | 2019,5,"[spark, kubernetes, terraform, natural language processing]",Targeting a Cybersecurity Engineer position to utilize my educational background and experience to drive results and achieve career objectives.,Current Title: Cybersecurity Engineer. Previous Titles: . Education: Bachelor's | Bachelor's in Electronics Engineering | Electronics Engineering | Pune University | 2019. Experience Years: 5. Ski...
2,R_0002,Prompt Engineer,Bachelor's | Bachelor's in Computer Science | Computer Science | Amity University | 2023,2,"[data analysis, node.js, machine learning, linux, jenkins, network security, rest apis]",Targeting a Prompt Engineer position to utilize my educational background and experience to drive results and achieve career objectives.,"Current Title: Prompt Engineer. Previous Titles: . Education: Bachelor's | Bachelor's in Computer Science | Computer Science | Amity University | 2023. Experience Years: 2. Skills: data analysis, ..."


In [7]:
#Sanity Checks
# Check for empty texts (should be very rare)
print("Empty job_text:", (jobs_clean["job_text"].str.len() == 0).sum())
print("Empty resume_text:", (resumes_clean["resume_text"].str.len() == 0).sum())

# Quick length stats
print("\njob_text length stats:")
print(jobs_clean["job_text"].str.len().describe())

print("\nresume_text length stats:")
print(resumes_clean["resume_text"].str.len().describe())


Empty job_text: 0
Empty resume_text: 0

job_text length stats:
count    1068.000000
mean      533.250936
std       160.962839
min       248.000000
25%       400.000000
50%       516.500000
75%       655.000000
max      1131.000000
Name: job_text, dtype: float64

resume_text length stats:
count    1200.000000
mean      506.986667
std        53.042289
min       365.000000
25%       470.000000
50%       507.000000
75%       542.250000
max       700.000000
Name: resume_text, dtype: float64


In [9]:
#Skill Vocabulary
all_skills = set()

for lst in jobs_clean["job_skills_list"]:
    all_skills.update(lst)

for lst in resumes_clean["resume_skills_list"]:
    all_skills.update(lst)

skills_vocab = sorted(all_skills)

print("Unique skills:", len(skills_vocab))
print("Sample skills:", skills_vocab[:40])

# Save vocab locally (small file)
vocab_path = DATA_PROCESSED / "skills_vocab.json"
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(skills_vocab, f, ensure_ascii=False, indent=2)

print("Saved:", vocab_path)


Unique skills: 1884
Sample skills: ['.net core', '.net framework', '3d design', '3d modeling', '3d modeling and animation expert', '3d modeling and graphics', 'a', 'accessibility', 'accessibility standards', 'accounting', 'active directory', 'active listening', 'actuator integration', 'adaptability', 'adaptable tone and style', 'adaptable writing style', 'adaptation of tone and style', 'adobe creative suite', 'adobe framemaker', 'adobe illustrator', 'adobe indesign', 'adobe photoshop', 'adobe xd', 'advanced 3d graphics', 'advanced 3d modeling', 'advanced 3d modeling and animation', 'advanced adobe photoshop', 'advanced ai', 'advanced ai and gameplay', 'advanced ai and gameplay systems', 'advanced analytics', 'advanced analytics (google analytics', 'advanced analytics and a', 'advanced bi modeling', 'advanced budgeting and resource allocation', 'advanced c', 'advanced c programming', 'advanced c++', 'advanced cad', 'advanced content optimization']
Saved: C:\Users\COMPUTER CARE\aeej1\Job

In [10]:
jobs_out = DATA_PROCESSED / "jobs_clean.parquet"
resumes_out = DATA_PROCESSED / "resumes_clean.parquet"

jobs_clean.to_parquet(jobs_out, index=False)
resumes_clean.to_parquet(resumes_out, index=False)

print("Saved:", jobs_out)
print("Saved:", resumes_out)


Saved: C:\Users\COMPUTER CARE\aeej1\JobPlatform\data\processed\jobs_clean.parquet
Saved: C:\Users\COMPUTER CARE\aeej1\JobPlatform\data\processed\resumes_clean.parquet
